<a href="https://colab.research.google.com/github/71percentbanana/gridathon/blob/main/Gridlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install catboost -q
!pip install pygeohash -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [19]:
import pandas as pd
import numpy as np
import pygeohash as pgh

from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# =========================
# CLEANING
# =========================

for data in [df, test_df]:
    data['Temperature'] = data['Temperature'].fillna(df['Temperature'].median())
    data['RoadType'] = data['RoadType'].fillna('Unknown')
    data['Weather'] = data['Weather'].fillna('Unknown')

# =========================
# TIME FEATURES
# =========================

for data in [df, test_df]:

    data[['hour', 'minute']] = (
        data['timestamp']
        .str.split(':', expand=True)
    )

    data['hour'] = data['hour'].astype(int)
    data['minute'] = data['minute'].astype(int)

    data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
    data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)

    data['minute_sin'] = np.sin(2 * np.pi * data['minute'] / 60)
    data['minute_cos'] = np.cos(2 * np.pi * data['minute'] / 60)

# =========================
# FEATURE CROSSES
# =========================

for data in [df, test_df]:

    data['RoadType_Lanes'] = (
        data['RoadType'].astype(str)
        + "_"
        + data['NumberofLanes'].astype(str)
    )

    data['Weather_Road'] = (
        data['Weather'].astype(str)
        + "_"
        + data['RoadType'].astype(str)
    )

    # data['Weather_Hour'] = (
    #     data['Weather'].astype(str)
    #     + "_"
    #     + data['hour'].astype(str)
    # )

    # data['Road_Hour'] = (
    #     data['RoadType'].astype(str)
    #     + "_"
    #     + data['hour'].astype(str)
    # )

# =========================
# GEOHASH FEATURES
# =========================

for data in [df, test_df]:

    data['geohash_4'] = data['geohash'].str[:4]
    data['geohash_5'] = data['geohash'].str[:5]
    data['geohash_6'] = data['geohash'].str[:6]

    data['latitude'] = data['geohash'].apply(
        lambda x: pgh.decode(x)[0]
    )

    data['longitude'] = data['geohash'].apply(
        lambda x: pgh.decode(x)[1]
    )

    # data['lat_sq'] = data['latitude'] ** 2
    # data['lon_sq'] = data['longitude'] ** 2
    # data['lat_lon'] = data['latitude'] * data['longitude']

# =========================
# FEATURES
# =========================

features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',

    'latitude',
    'longitude',

    'day',

    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',

    'hour_sin',
    'hour_cos',
    'minute_sin',
    'minute_cos',

    'RoadType_Lanes',
    'Weather_Road',
]

cat_features = [
    'geohash_4',
    'geohash_5',
    'geohash_6',

    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather',

    'RoadType_Lanes',
    'Weather_Road',
]



X = df[features].copy()
y = df['demand']
test_X = test_df[features].copy()

# =========================
# CV
# =========================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

    print(f"Training Fold {fold + 1}")

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # =====================
    # OOF TARGET ENCODING
    # =====================


    # =====================
    # MODEL
    # =====================

    model = CatBoostRegressor(
        iterations=2500,
        depth=8,
        learning_rate=0.03,
        loss_function='RMSE',
        random_strength=1,
        l2_leaf_reg=5,
        bagging_temperature=0.7,
        verbose=0,
        early_stopping_rounds=200
    )
    model.fit(
      X_train,
      y_train,
      eval_set=(X_val, y_val),
      cat_features=cat_features,
      verbose=100
    )
    val_preds = model.predict(X_val)

    oof_preds[val_idx] = val_preds

    test_preds += (
        model.predict(test_X) / 5
    )

# =========================
# METRICS
# =========================

rmse = np.sqrt(
    mean_squared_error(y, oof_preds)
)

r2 = r2_score(
    y,
    oof_preds
)

print("\nFINAL RESULTS")
print("OOF RMSE:", rmse)
print("OOF R2:", r2)

# =========================
# SUBMISSION
# =========================

test_preds = np.clip(test_preds, 0, 1)

submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_preds
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

Training Fold 1
0:	learn: 0.1389179	test: 0.1389134	best: 0.1389134 (0)	total: 167ms	remaining: 6m 57s
100:	learn: 0.0463457	test: 0.0450182	best: 0.0450182 (100)	total: 11.9s	remaining: 4m 43s
200:	learn: 0.0410517	test: 0.0404351	best: 0.0404351 (200)	total: 22.2s	remaining: 4m 13s
300:	learn: 0.0395044	test: 0.0391793	best: 0.0391793 (300)	total: 33.6s	remaining: 4m 5s
400:	learn: 0.0384154	test: 0.0383088	best: 0.0383088 (400)	total: 43.4s	remaining: 3m 47s
500:	learn: 0.0375644	test: 0.0375961	best: 0.0375961 (500)	total: 54.8s	remaining: 3m 38s
600:	learn: 0.0369701	test: 0.0371545	best: 0.0371545 (600)	total: 1m 7s	remaining: 3m 32s
700:	learn: 0.0359321	test: 0.0363581	best: 0.0363581 (700)	total: 1m 21s	remaining: 3m 29s
800:	learn: 0.0349963	test: 0.0356514	best: 0.0356514 (800)	total: 1m 35s	remaining: 3m 23s
900:	learn: 0.0342007	test: 0.0350393	best: 0.0350386 (899)	total: 1m 49s	remaining: 3m 14s
1000:	learn: 0.0334835	test: 0.0345136	best: 0.0345136 (999)	total: 2m 3s	re

In [16]:
print(df['geohash_6'].nunique())
print(df['geohash_5'].nunique())
print(df['geohash_4'].nunique())
print(df.shape)
print(df.size)
print(test_df.shape)

1249
56
6
(77299, 30)
(41778, 29)
